In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import (
    coint,
    adfuller
)

import warnings

warnings.filterwarnings("ignore")


data = pd.read_csv(
    "nifty_50_comp_price_data.csv",
    index_col=0,
    parse_dates=True
)

data.head()

,ADANIENT.NS,ADANIPORTS.NS,APOLLOHOSP.NS,ASIANPAINT.NS,AXISBANK.NS,BAJAJ-AUTO.NS,BAJFINANCE.NS,BEL.NS,BHARTIARTL.NS,CIPLA.NS,...,SHRIRAMFIN.NS,SUNPHARMA.NS,TATACONSUM.NS,TATASTEEL.NS,TCS.NS,TECHM.NS,TITAN.NS,TRENT.NS,ULTRACEMCO.NS,WIPRO.NS
Date,,,,,,,,,,,,,,,,,,,,,
2020-01-01,205.794525,361.062561,1403.106934,1690.746338,745.586914,2599.727783,413.532196,29.713722,433.316498,449.420258,...,201.816833,406.074036,306.714355,38.457085,1841.149292,602.617493,1132.704956,526.402710,3942.765625,113.709183
2020-01-02,209.111359,366.321014,1470.294067,1688.342407,753.802612,2575.711670,414.973724,30.603800,435.132721,447.153778,...,204.996552,406.681732,306.240814,39.862995,1832.697754,605.740845,1133.538574,538.030945,4117.158691,113.984634
2020-01-03,206.240067,365.699554,1461.883301,1651.334717,739.860840,2535.314453,409.833008,29.951080,435.037109,443.801361,...,204.899414,415.704620,301.363312,39.768440,1869.222168,612.897095,1117.942017,532.614319,4092.329102,115.269989
2020-01-06,197.576599,363.500580,1438.667969,1609.613159,720.242676,2506.924316,390.604340,28.764307,429.827362,440.779388,...,197.338669,411.356781,295.586090,38.909275,1869.051880,609.180664,1136.481201,526.700928,4032.095703,115.751999
2020-01-07,202.032074,367.898529,1454.603760,1625.877197,722.732300,2507.171875,391.674469,28.586292,425.477966,442.526489,...,198.601746,417.387543,298.237915,39.143593,1873.639404,614.478577,1137.805542,529.682434,4114.540039,117.152138


In [30]:
returns = data.pct_change()

In [31]:
corr_matrix = returns.corr()

candidate_pairs = []

for i in range(len(corr_matrix.columns)):

    for j in range(i + 1,
                   len(corr_matrix.columns)):

        stock1 = corr_matrix.columns[i]
        stock2 = corr_matrix.columns[j]

        corr = corr_matrix.iloc[i, j]

        if corr > 0.40:

            candidate_pairs.append(
                [
                    stock1,
                    stock2,
                    corr
                ]
            )

candidate_pairs = pd.DataFrame(
    candidate_pairs,
    columns=[
        "Stock1",
        "Stock2",
        "Correlation"
    ]
)

print(
    "Candidate Pairs:",
    len(candidate_pairs)
)

candidate_pairs.head()

Candidate Pairs: 215


,Stock1,Stock2,Correlation
0,ADANIENT.NS,ADANIPORTS.NS,0.702799
1,ADANIENT.NS,JIOFIN.NS,0.466130
2,ADANIENT.NS,SBIN.NS,0.421570
3,ADANIPORTS.NS,BEL.NS,0.422913
4,ADANIPORTS.NS,GRASIM.NS,0.434866


In [32]:
results = []

for _, row in candidate_pairs.iterrows():

    stock1 = row["Stock1"]
    stock2 = row["Stock2"]

    try:

        score, coint_pvalue, _ = coint(
            data[stock1],
            data[stock2]
        )

        beta = np.polyfit(
            data[stock2],
            data[stock1],
            1
        )[0]

        spread = (
            data[stock1]
            - beta * data[stock2]
        )

        adf_pvalue = adfuller(
            spread.dropna()
        )[1]

        results.append(
            [
                stock1,
                stock2,
                row["Correlation"],
                coint_pvalue,
                adf_pvalue
            ]
        )

    except:

        continue

In [33]:
pairs_df = pd.DataFrame(
    results,
    columns=[
        "Stock1",
        "Stock2",
        "Correlation",
        "Cointegration_P",
        "ADF_P"
    ]
)

pairs_df = pairs_df.sort_values(
    [
        "Cointegration_P",
        "ADF_P"
    ]
)


pairs_df.head(35)

,Stock1,Stock2,Correlation,Cointegration_P,ADF_P
68,COALINDIA.NS,NTPC.NS,0.551356,0.000330,0.000047
25,AXISBANK.NS,LT.NS,0.498352,0.001585,0.000263
191,NTPC.NS,ONGC.NS,0.483397,0.003396,0.000612
133,HINDALCO.NS,SBIN.NS,0.460290,0.003445,0.000622
32,AXISBANK.NS,TITAN.NS,0.401480,0.003950,0.000720
17,AXISBANK.NS,GRASIM.NS,0.513988,0.004259,0.000787
192,NTPC.NS,POWERGRID.NS,0.598073,0.005931,0.001141
45,BAJFINANCE.NS,KOTAKBANK.NS,0.525920,0.007243,0.001432
31,AXISBANK.NS,TATASTEEL.NS,0.400063,0.007542,0.001495
86,GRASIM.NS,ICICIBANK.NS,0.500039,0.010078,0.002083


In [34]:
strong_pairs = pairs_df[
    (pairs_df["Cointegration_P"] < 0.05)
    &
    (pairs_df["ADF_P"] < 0.05)
]

strong_pairs

,Stock1,Stock2,Correlation,Cointegration_P,ADF_P
68,COALINDIA.NS,NTPC.NS,0.551356,0.000330,0.000047
25,AXISBANK.NS,LT.NS,0.498352,0.001585,0.000263
191,NTPC.NS,ONGC.NS,0.483397,0.003396,0.000612
133,HINDALCO.NS,SBIN.NS,0.460290,0.003445,0.000622
32,AXISBANK.NS,TITAN.NS,0.401480,0.003950,0.000720
17,AXISBANK.NS,GRASIM.NS,0.513988,0.004259,0.000787
192,NTPC.NS,POWERGRID.NS,0.598073,0.005931,0.001141
45,BAJFINANCE.NS,KOTAKBANK.NS,0.525920,0.007243,0.001432
31,AXISBANK.NS,TATASTEEL.NS,0.400063,0.007542,0.001495
86,GRASIM.NS,ICICIBANK.NS,0.500039,0.010078,0.002083


In [35]:
len(strong_pairs)

30

In [36]:
top_pairs = [
    ("COALINDIA.NS", "NTPC.NS"),
    ("NTPC.NS", "ONGC.NS"),
    ("NTPC.NS", "POWERGRID.NS"),
    ("HDFCBANK.NS", "KOTAKBANK.NS"),
    ("M&M.NS", "MARUTI.NS")
]

In [37]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

from statsmodels.tsa.stattools import (
    coint,
    adfuller
)

results = []

for stock1, stock2 in top_pairs:

    score, coint_pvalue, _ = coint(
        data[stock1],
        data[stock2]
    )

    x = sm.add_constant(
        data[stock2]
    )

    model = sm.OLS(
        data[stock1],
        x
    ).fit()

    beta = model.params[stock2]

    spread = (
        data[stock1]
        - beta * data[stock2]
    )

    adf_pvalue = adfuller(
        spread.dropna()
    )[1]

    zscore = (
        spread - spread.mean()
    ) / spread.std()

    entries = (
        ((zscore.shift(1) <= 2) & (zscore > 2))
        |
        ((zscore.shift(1) >= -2) & (zscore < -2))
    ).sum()

    results.append([
        stock1,
        stock2,
        round(coint_pvalue, 6),
        round(adf_pvalue, 6),
        int(entries)
    ])

summary = pd.DataFrame(
    results,
    columns=[
        "Stock1",
        "Stock2",
        "Cointegration_P",
        "ADF_P",
        "Entries"
    ]
)

summary.sort_values(
    "Cointegration_P"
)

,Stock1,Stock2,Cointegration_P,ADF_P,Entries
0,COALINDIA.NS,NTPC.NS,0.000330,0.000047,26
1,NTPC.NS,ONGC.NS,0.003396,0.000612,14
2,NTPC.NS,POWERGRID.NS,0.005931,0.001141,16
3,HDFCBANK.NS,KOTAKBANK.NS,0.011228,0.002354,15
4,M&M.NS,MARUTI.NS,0.038603,0.009853,8


In [38]:
import statsmodels.api as sm

stock1 = "COALINDIA.NS"
stock2 = "NTPC.NS"

x = sm.add_constant(data[stock2])

model = sm.OLS(
    data[stock1],
    x
).fit()

beta = model.params[stock2]

spread = (
    data[stock1]
    - beta * data[stock2]
)

zscore = (
    spread - spread.mean()
) / spread.std()

print("Beta:", beta)

Beta: 1.172788562773483


In [39]:
position = pd.Series(
    0,
    index=zscore.index
)

current_position = 0

for i in range(len(zscore)):

    z = zscore.iloc[i]

    if current_position == 0:

        if z > 2:
            current_position = -1

        elif z < -2:
            current_position = 1

    elif current_position == 1:

        if z >= 0:
            current_position = 0

    elif current_position == -1:

        if z <= 0:
            current_position = 0

    position.iloc[i] = current_position

In [40]:
trade_log = []

current_position = 0
entry_date = None
entry_z = None

for i in range(1, len(position)):

    prev_pos = position.iloc[i - 1]
    curr_pos = position.iloc[i]

    date = position.index[i]

    if prev_pos == 0 and curr_pos != 0:

        entry_date = date
        entry_z = zscore.iloc[i]

    elif prev_pos != 0 and curr_pos == 0:

        exit_date = date
        exit_z = zscore.iloc[i]

        trade_log.append([
            entry_date,
            exit_date,
            "Long Spread" if prev_pos == 1 else "Short Spread",
            entry_z,
            exit_z,
            (exit_date - entry_date).days
        ])

In [41]:
trade_log = pd.DataFrame(
    trade_log,
    columns=[
        "Entry_Date",
        "Exit_Date",
        "Direction",
        "Entry_Z",
        "Exit_Z",
        "Holding_Days"
    ]
)

trade_log

,Entry_Date,Exit_Date,Direction,Entry_Z,Exit_Z,Holding_Days
0,2023-07-31,2023-10-10,Long Spread,-2.449524,0.196666,71
1,2024-02-07,2024-04-08,Short Spread,2.100312,-0.202739,61
2,2024-05-22,2024-06-27,Short Spread,2.063660,-0.076443,36
3,2024-07-16,2024-08-05,Short Spread,2.193529,-0.415560,20
4,2024-08-23,2024-09-05,Short Spread,2.216529,-0.203911,13
5,2024-09-19,2025-01-13,Long Spread,-2.725499,0.366004,116
6,2025-03-25,2025-06-02,Long Spread,-2.340118,0.204792,69
7,2026-01-12,2026-02-11,Short Spread,2.736592,-0.286991,30
8,2026-04-02,2026-04-10,Short Spread,2.341138,-0.120905,8


In [42]:
print(
    "Completed Trades:",
    len(trade_log)
)

print(
    "Average Holding Days:",
    trade_log["Holding_Days"].mean()
)

Completed Trades: 9
Average Holding Days: 47.111111111111114


In [43]:
axis_ret = data["COALINDIA.NS"].pct_change()

ntpc_ret = data["NTPC.NS"].pct_change()

spread_ret = (
    axis_ret
    - beta * ntpc_ret
)

strategy_ret = (
    position.shift(1)
    * spread_ret
)

In [44]:
equity_curve = (
    1 + strategy_ret.fillna(0)
).cumprod()

total_return = (
    equity_curve.iloc[-1] - 1
)

years = (
    len(strategy_ret)
    / 252
)

cagr = (
    equity_curve.iloc[-1]
) ** (1 / years) - 1

sharpe = (
    strategy_ret.mean()
    / strategy_ret.std()
) * np.sqrt(252)

rolling_max = equity_curve.cummax()

drawdown = (
    equity_curve
    / rolling_max
    - 1
)

max_dd = drawdown.min()

print(
    f"Total Return: {total_return:.2%}"
)

print(
    f"CAGR: {cagr:.2%}"
)

print(
    f"Sharpe: {sharpe:.2f}"
)

print(
    f"Max Drawdown: {max_dd:.2%}"
)

Total Return: 186.36%
CAGR: 17.94%
Sharpe: 1.47
Max Drawdown: -7.91%


In [45]:
wins = (
    strategy_ret[strategy_ret > 0]
)

losses = (
    strategy_ret[strategy_ret < 0]
)

profit_factor = (
    wins.sum()
    /
    abs(losses.sum())
)

win_rate = (
    (strategy_ret > 0).mean()
)

print(
    f"Win Rate: {win_rate:.2%}"
)

print(
    f"Profit Factor: {profit_factor:.2f}"
)

Win Rate: 11.20%
Profit Factor: 1.79


In [46]:
trade_returns = []

for _, trade in trade_log.iterrows():

    entry_date = trade["Entry_Date"]
    exit_date = trade["Exit_Date"]

    trade_ret = (
        1
        + strategy_ret.loc[
            entry_date:exit_date
        ]
    ).prod() - 1

    trade_returns.append(
        trade_ret
    )

trade_log["Trade_Return"] = trade_returns

trade_log

,Entry_Date,Exit_Date,Direction,Entry_Z,Exit_Z,Holding_Days,Trade_Return
0,2023-07-31,2023-10-10,Long Spread,-2.449524,0.196666,71,0.204449
1,2024-02-07,2024-04-08,Short Spread,2.100312,-0.202739,61,0.109944
2,2024-05-22,2024-06-27,Short Spread,2.063660,-0.076443,36,0.089556
3,2024-07-16,2024-08-05,Short Spread,2.193529,-0.415560,20,0.111746
4,2024-08-23,2024-09-05,Short Spread,2.216529,-0.203911,13,0.086990
5,2024-09-19,2025-01-13,Long Spread,-2.725499,0.366004,116,0.166678
6,2025-03-25,2025-06-02,Long Spread,-2.340118,0.204792,69,0.120467
7,2026-01-12,2026-02-11,Short Spread,2.736592,-0.286991,30,0.137326
8,2026-04-02,2026-04-10,Short Spread,2.341138,-0.120905,8,0.100734


In [47]:
wins = trade_log[
    trade_log["Trade_Return"] > 0
]

losses = trade_log[
    trade_log["Trade_Return"] < 0
]

win_rate = (
    len(wins)
    / len(trade_log)
)

avg_winner = wins[
    "Trade_Return"
].mean()

avg_loser = losses[
    "Trade_Return"
].mean()

profit_factor = (
    wins["Trade_Return"].sum()
    /
    abs(
        losses["Trade_Return"].sum()
    )
)

In [48]:
print(
    f"Completed Trades: {len(trade_log)}"
)

print(
    f"Trade Win Rate: {win_rate:.2%}"
)

print(
    f"Average Winner: {avg_winner:.2%}"
)

print(
    f"Average Loser: {avg_loser:.2%}"
)

print(
    f"Profit Factor: {profit_factor:.2f}"
)

print(
    f"Average Holding Days: {trade_log['Holding_Days'].mean():.1f}"
)

Completed Trades: 9
Trade Win Rate: 100.00%
Average Winner: 12.53%
Average Loser: nan%
Profit Factor: inf
Average Holding Days: 47.1
